Nikolaj Skou-Larsen - gkm406

In [14]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from maketables import dtable
from lets_plot import *
LetsPlot.setup_html()
df = pd.read_stata("A1_kommune.dta")

df.columns


Index(['nr', 'kommune', 'taxrev', 'taxrate', 'pop'], dtype='str')

# Problem 1

#### Tabel 1

In [15]:
# Mean + std af variablerne 
vars = ['taxrev','taxrate','pop']
labels = {
    'taxrev': 'Tax revenue (Mio DKK)',
    'taxrate' : 'Tax rate (%)',
    'pop' : 'Population'
}

dtable.DTable(
    df,
    vars,
    labels=labels,
    counts_row_below=True,
    stats=['mean_newline_std']
)

<maketables.mtable.MTable.__repr__.<locals>.DualOutput at 0x1ff2d7a17f0>

#### Tabel 2

In [16]:
#Describtive analysis
vars =['taxrev','taxrate','pop']
print(df.groupby('kommune')[vars].mean().round(3))

# Max/min taxrev and taxrate
max_tax_rev = df.loc[df['taxrev'].idxmax()]

min_tax_rev = df.loc[df['taxrev'].idxmin()]

max_tax_rate = df.loc[df['taxrate'].idxmax()]

min_tax_rate = df.loc[df['taxrate'].idxmin()]


# Export a table to Latex
selected = pd.DataFrame([
    max_tax_rev,
    min_tax_rev,
    max_tax_rate,
    min_tax_rate
])
selected = selected.rename(columns={
    'taxrev': 'taxrev (mio. DKK)',
    'taxrate': 'taxrate (%)',
    'pop': 'population'
})
selected['kommune'] = [
    'Max tax revenue: ' + str(max_tax_rev['kommune']),
    'Min tax revenue: ' + str(min_tax_rev['kommune']),
    'Max tax rate: ' + str(max_tax_rate['kommune']),
    'Min tax rate: ' + str(min_tax_rate['kommune'])
]
selected['kommune'] = selected['kommune'].str.replace(' Kommune', '', regex=False)
latex_df = selected.copy()


latex_df = latex_df.rename(columns={
    'kommune': 'Municipality',
    'taxrev (mio. DKK)': 'Tax revenue (mio. DKK)',
    'taxrate (%)': 'Tax rate (\%)',
    'population': 'Population'
})

latex_df['Tax revenue (mio. DKK)'] = latex_df['Tax revenue (mio. DKK)'].map(
    lambda x: f"{x:,.1f}"
)

latex_df['Tax rate (\\%)'] = latex_df['Tax rate (\\%)'].map(
    lambda x: f"{x:.1f}"
)

latex_df['Population'] = latex_df['Population'].map(
    lambda x: f"{x:,.0f}"
)

latex_code = latex_df.to_latex(
    index=False,
    escape=False,
    caption="Maximum and minimum values of tax revenue and tax rate",
    label="tab:descriptive"
)

print(latex_code)


                               taxrev    taxrate       pop
kommune                                                   
Aabenraa Kommune          4547.832031  25.400000   59978.0
Aalborg Kommune          16330.091797  25.400000  197426.0
Aarhus Kommune           26152.337891  24.400000  306650.0
Albertslund Kommune       2964.559082  24.600000   27730.0
Allerød Kommune           1613.119995  25.299999   24089.0
...                               ...        ...       ...
Vejle Kommune             8517.554688  23.400000  106383.0
Vesthimmerlands Kommune   3324.785889  27.200001   38106.0
Viborg Kommune            6772.346191  25.799999   93310.0
Vordingborg Kommune       3733.774902  24.900000   46319.0
Ærø Kommune                566.492981  26.100000    6679.0

[98 rows x 3 columns]
\begin{table}
\caption{Maximum and minimum values of tax revenue and tax rate}
\label{tab:descriptive}
\begin{tabular}{rllll}
\toprule
nr & Municipality & Tax revenue (mio. DKK) & Tax rate (\%) & Population \\


# Problem 2

#### Tabel 3

In [17]:
#OLS for log-level
df = pd.read_stata("A1_kommune.dta")
df['log_taxrev'] = np.log(df.taxrev)


df['const'] = 1


result = sm.OLS(df['log_taxrev'], df[['const','taxrate']]).fit()

table = pd.DataFrame({
    'Coefficient': result.params,
    'Std. Error': result.bse,
    'R-squared' : result.rsquared
})

# Convert to LaTeX
latex_table = table.to_latex(
    float_format="%.4f",
    caption="OLS Regression Results",
    label="tab:ols",
    escape=False
)
print(latex_table)

\begin{table}
\caption{OLS Regression Results}
\label{tab:ols}
\begin{tabular}{lrrr}
\toprule
 & Coefficient & Std. Error & R-squared \\
\midrule
const & 11.6982 & 2.1430 & 0.0285 \\
taxrate & -0.1426 & 0.0850 & 0.0285 \\
\bottomrule
\end{tabular}
\end{table}



Negativ beta1 kan type på at højere skat får folk til at arbejde mindere og dermed bliver skatteindtægterne lavere. Laffer kurven? 

#### Tabel 4

In [18]:

df['log_pop'] = np.log(df['pop'].values)
df['const'] = 1


y = df['log_taxrev']
X = df[['const','taxrate', 'log_pop']]


result = sm.OLS(y, X).fit()

# Create table with coefficients and standard errors
table = pd.DataFrame({
    'Coefficient': result.params,
    'Std. Error': result.bse,
    'R-squared' : result.rsquared
})

# Convert to LaTeX
latex_table = table.to_latex(
    float_format="%.4f",
    caption="OLS Regression Results",
    label="tab:ols",
    escape=False
)
print(latex_table)


\begin{table}
\caption{OLS Regression Results}
\label{tab:ols}
\begin{tabular}{lrrr}
\toprule
 & Coefficient & Std. Error & R-squared \\
\midrule
const & -2.8022 & 0.3756 & 0.9801 \\
taxrate & 0.0226 & 0.0125 & 0.9801 \\
log_pop & 0.9711 & 0.0144 & 0.9801 \\
\bottomrule
\end{tabular}
\end{table}



# Problem 3

### Tabel 6 & 7

In [ ]:
df['log_pop'] = np.log(df['pop'].values)
df['const'] = 1


# First OLS regression, taxrate on log(pop)
result1 = sm.OLS(df['taxrate'], df[['const','log_pop']]).fit()
df['res1'] = result1.resid

# Second OLS regression, log(taxrev) on residuals from first OLS 
result2 = sm.OLS(df['log_taxrev'], df[['const','res1']]).fit()

# Create table with coefficients and standard errors
table = pd.DataFrame({
    'Coefficient': result1.params,
    'Std. Error': result1.bse,
    'R-squared' : result1.rsquared
})

# Convert to LaTeX
latex_table = table.to_latex(
    float_format="%.4f",
    caption="OLS Regression Results",
    label="tab:ols",
    escape=False
)

print(latex_table)

# Create table with coefficients and standard errors
table = pd.DataFrame({
    'Coefficient': result2.params,
    'Std. Error': result2.bse,
    'R-squared' : result2.rsquared
})

# Convert to LaTeX
latex_table = table.to_latex(
    float_format="%.4f",
    caption="OLS Regression Results",
    label="tab:ols",
    escape=False
)

print(latex_table)

\begin{table}
\caption{OLS Regression Results}
\label{tab:ols}
\begin{tabular}{lrrr}
\toprule
 & Coefficient & Std. Error & R-squared \\
\midrule
const & 27.6268 & 1.2341 & 0.0387 \\
log_pop & -0.2273 & 0.1156 & 0.0387 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}
\caption{OLS Regression Results}
\label{tab:ols}
\begin{tabular}{lrrr}
\toprule
 & Coefficient & Std. Error & R-squared \\
\midrule
const & 8.1031 & 0.0778 & 0.0007 \\
res1 & 0.0226 & 0.0879 & 0.0007 \\
\bottomrule
\end{tabular}
\end{table}

